# Decline Curve Analysis with petropt

This notebook demonstrates Arps decline curve analysis using `petropt`.
You'll learn:
1. The three Arps decline models
2. How to forecast production
3. How to calculate EUR (Estimated Ultimate Recovery)
4. Comparing exponential vs hyperbolic vs harmonic decline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from petropt.correlations.decline import arps_decline, arps_cumulative, arps_eur

## 1. Arps Decline Models

J.J. Arps (1945) proposed three empirical decline models based on the **b-factor**:

| Model | b-factor | Equation |
|-------|----------|----------|
| Exponential | b = 0 | q(t) = qi × exp(-Di × t) |
| Hyperbolic | 0 < b < 1 | q(t) = qi / (1 + b×Di×t)^(1/b) |
| Harmonic | b = 1 | q(t) = qi / (1 + Di × t) |

In [ ]:
# Common parameters
qi = 1000    # Initial rate: 1000 STB/day
di = 0.05    # Decline rate: 5%/month
t = np.linspace(0, 60, 100)  # 60 months (5 years)

# Calculate rates for each model
q_exp = arps_decline(qi=qi, di=di, b=0, t=t)       # Exponential
q_hyp = arps_decline(qi=qi, di=di, b=0.5, t=t)     # Hyperbolic (b=0.5)
q_har = arps_decline(qi=qi, di=di, b=1.0, t=t)     # Harmonic

plt.figure(figsize=(10, 6))
plt.plot(t, q_exp, 'b-', linewidth=2, label='Exponential (b=0)')
plt.plot(t, q_hyp, 'r-', linewidth=2, label='Hyperbolic (b=0.5)')
plt.plot(t, q_har, 'g-', linewidth=2, label='Harmonic (b=1)')
plt.xlabel('Time (months)')
plt.ylabel('Rate (STB/day)')
plt.title('Arps Decline Models Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0)
plt.tight_layout()
plt.show()

## 2. Effect of b-Factor

The b-factor controls the **curvature** of the decline. Higher b = slower decline.

In [ ]:
plt.figure(figsize=(10, 6))

for b in [0, 0.2, 0.5, 0.8, 1.0, 1.5]:
    q = arps_decline(qi=qi, di=di, b=b, t=t)
    plt.plot(t, q, linewidth=2, label=f'b = {b}')

plt.xlabel('Time (months)')
plt.ylabel('Rate (STB/day)')
plt.title('Effect of b-Factor on Decline Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0)
plt.tight_layout()
plt.show()

## 3. Cumulative Production

Track total production over time for each decline model.

In [ ]:
t_months = np.arange(1, 61)  # 1 to 60 months

np_exp = [arps_cumulative(qi=qi, di=di, b=0, t=t_val) for t_val in t_months]
np_hyp = [arps_cumulative(qi=qi, di=di, b=0.5, t=t_val) for t_val in t_months]
np_har = [arps_cumulative(qi=qi, di=di, b=1.0, t=t_val) for t_val in t_months]

plt.figure(figsize=(10, 6))
plt.plot(t_months, np_exp, 'b-', linewidth=2, label='Exponential (b=0)')
plt.plot(t_months, np_hyp, 'r-', linewidth=2, label='Hyperbolic (b=0.5)')
plt.plot(t_months, np_har, 'g-', linewidth=2, label='Harmonic (b=1)')
plt.xlabel('Time (months)')
plt.ylabel('Cumulative Production (STB-months)')
plt.title('Cumulative Production by Decline Model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"5-year cumulative (Exponential): {np_exp[-1]:,.0f} STB-months")
print(f"5-year cumulative (Hyperbolic):  {np_hyp[-1]:,.0f} STB-months")
print(f"5-year cumulative (Harmonic):    {np_har[-1]:,.0f} STB-months")

## 4. EUR Calculation

Estimated Ultimate Recovery — integrate until economic limit is reached.

In [ ]:
economic_limit = 10  # STB/day

for b_val, name in [(0, 'Exponential'), (0.5, 'Hyperbolic'), (1.0, 'Harmonic')]:
    result = arps_eur(qi=qi, di=di, b=b_val, economic_limit=economic_limit)
    print(f"{name} (b={b_val}):")
    print(f"  EUR: {result['eur']:,.0f} STB-months")
    print(f"  Time to limit: {result['time_to_limit']:.1f} months ({result['time_to_limit']/12:.1f} years)")
    print(f"  Final rate: {result['final_rate']:.1f} STB/day")
    print()

> **Fitting decline curves manually is tedious.** [petropt-suite](https://petropt.com/suite) provides automated Bayesian DCA fitting with P10/P50/P90 uncertainty bounds, multi-segment decline, and probability forecasts. Try the free online version at [tools.petropt.com/decline-curve](https://tools.petropt.com/decline-curve/).

## 5. Fitting to Real Data

Use the sample production data to demonstrate decline analysis.

## Key Takeaways

1. **Exponential decline** (b=0) gives the most conservative EUR estimate
2. **Hyperbolic decline** (0<b<1) is the most common in practice
3. **Harmonic decline** (b=1) gives the most optimistic EUR estimate
4. The **b-factor** must be constrained — unbounded b leads to unrealistic forecasts
5. Always compare multiple models and use engineering judgment

## Next Steps

- **Free online tools**: [tools.petropt.com](https://tools.petropt.com) — interactive decline curve calculator
- **Advanced DCA**: [petropt.com/suite](https://petropt.com/suite) — Bayesian fitting, type curves, multi-well batch analysis
- **More correlations**: See `student_intro.ipynb` for PVT, IPR, and Z-factor

## Key Takeaways

1. **Exponential decline** (b=0) gives the most conservative EUR estimate
2. **Hyperbolic decline** (0<b<1) is the most common in practice
3. **Harmonic decline** (b=1) gives the most optimistic EUR estimate
4. The **b-factor** must be constrained — unbounded b leads to unrealistic forecasts
5. Always compare multiple models and use engineering judgment

Learn more at [petropt.com](https://petropt.com)